**Link al repositorio de GitHub:** [https://github.com/vgcarlol/Aprendizaje-Por-Refuerzo](https://github.com/vgcarlol/Aprendizaje-Por-Refuerzo)

# CC3104 - Aprendizaje por Refuerzo
## Laboratorio 1 - Entrega Parcial

---

### Task 1: Diseño formal del MDP

#### 1. Espacio de estados $\mathcal{S}$
Para que la propiedad de Markov se satisfaga razonablemente, el estado debe contener la información mínima necesaria para decidir la siguiente acción sin depender del historial pasado. El estado $S_t$ se compondrá de la tupla: $(x, y, P, B)$
*   **$(x, y)$ Posición actual:** Coordenadas en la cuadrícula 5x5. *Justificación:* Indispensable para saber dónde está el drone espacialmente.
*   **$P$ Estado del paquete:** Booleano (1 = a bordo, 0 = entregado). *Justificación:* Determina si la meta actual es dirigirse al punto de entrega o regresar a la base central.
*   **$B$ Nivel de batería:** Discretizado (ej. 0 a 100%). *Justificación:* Crítico para tomar decisiones de retorno de emergencia y evitar caídas.
*   **Omisiones deliberadas:** Velocidad del viento, temperatura de los motores, obstáculos dinámicos temporales. *Consecuencia:* Al omitirlos, la propiedad de Markov se debilita levemente porque el éxito de un movimiento dependerá de factores climáticos invisibles para el estado, introduciendo aleatoriedad estocástica en las transiciones en lugar de determinismo.

#### 2. Espacio de acciones $\mathcal{A}$
El espacio de acciones es **discreto**, ya que el entorno es una cuadrícula finita. 
*   $\mathcal{A} = \{\text{Norte, Sur, Este, Oeste, Entregar, Esperar}\}$
*   **Restricciones:** En los límites del grid (ej. en el borde superior), la acción de moverse fuera de la cuadrícula debería estar restringida. Asimismo, la acción "Entregar" solo debe tener sentido si el drone está en la celda de destino y $P=1$.
*   **Modelado en MDP:** Estas restricciones se modelan mediante la **función de transición y recompensa**. Si el drone intenta moverse contra una pared, la función de transición lo deja en el mismo estado con probabilidad 1, y la función de recompensa puede aplicar una ligera penalización por chocar o gastar tiempo inútilmente.

#### 3. Función de recompensa $R(s, a, s')$
El diseño debe balancear entrega, batería y seguridad:
*   **Eficiencia:** $-1$ por cada paso (incentiva la ruta más corta).
*   **Entrega y Regreso:** $+100$ por hacer la acción "Entregar" en el destino correcto, y $+100$ por regresar a la base y terminar el episodio.
*   **Seguridad / Batería:** $-500$ si el nivel de batería $B$ llega a 0 antes de estar en la base (el dron se estrella), terminando el episodio inmediatamente.
*   **Ponderación:** La seguridad tiene el peso negativo más grande absoluto ($|-500| > 100$). Un dron estrellado cuesta muchísimo dinero y puede herir personas.
*   **Consecuencias de mala ponderación:** Si la penalización por paso ($-1$) fuera masiva (ej. $-200$), el dron aprendería que lo mejor es quedarse en la base sin salir para evitar acumular castigos rápidos (Negative side-effects). Si la penalización por estrellarse fuera muy pequeña, el dron tomaría misiones suicidas para intentar entregar el paquete a toda costa.

#### 4. Función de transición $P(s' | s, a)$
La función debe ser **estocástica**. Aunque un drone reciba el comando "Norte", en el dominio real existen fuentes de aleatoriedad como ráfagas de viento, aves, o pérdida momentánea de señal GPS.
*   **Transición 1 (Éxito esperado):** $P(s'=(x, y+1) \mid s=(x,y), a=\text{Norte}) = 0.8$. El drone logra avanzar hacia el norte porque las condiciones son ideales la mayor parte del tiempo.
*   **Transición 2 (Desvío por viento lateral):** $P(s'=(x+1, y) \mid s=(x,y), a=\text{Norte}) = 0.1$. Una ráfaga de viento empuja al drone a la celda del Este mientras intentaba ir al Norte.
*   **Transición 3 (Freno por viento en contra):** $P(s'=(x, y) \mid s=(x,y), a=\text{Norte}) = 0.1$. El viento de frente es tan fuerte que el drone gasta su turno intentando avanzar sin lograr salir de su celda actual.

#### 5. Factor de descuento $\gamma$
Se propone **$\gamma = 0.99$**.
*   **Justificación:** En la logística urbana, cumplir el objetivo a largo plazo (entregar el paquete y volver) es lo primordial, incluso si toma 15 o 20 pasos de vuelo. Un $\gamma$ muy cercano a 1 permite que la recompensa lejana de $+100$ (regreso a base) se propague fuertemente hacia los estados iniciales. Si usaramos un $\gamma=0.5$ (miope), el drone solo vería los costos inmediatos de dar un paso y las recompensas a 10 pasos de distancia tendrían un valor percibido cercano a cero.

---

### Task 2: Preguntas Teóricas

#### 1. Violaciones a la propiedad de Markov
*   **Situación 1 (Tráfico aéreo dinámico / Otros drones):** Si hay otros drones moviéndose, nuestro estado actual $(x,y)$ no basta para predecir colisiones, ya que las posiciones futuras dependen de las trayectorias de los demás. 
    *   *Extensión:* Incluir la posición de los demás drones en el estado.
    *   *Costo Computacional:* El espacio de estados sufriría una inmensa explosión combinatoria. Un grid de 5x5 (25 estados base) con 3 drones pasaría a tener $25^3 = 15,625$ combinaciones espaciales.
*   **Situación 2 (Degradación térmica de la batería):** Si la batería se calienta por uso prolongado, su tasa de descarga real aumenta. Conocer solo el nivel (ej. 50%) no es suficiente; importa *cómo* llegó a 50%.
    *   *Extensión:* Añadir una variable que represente la "Temperatura de motores" o "Tiempo continuo de vuelo".
    *   *Costo Computacional:* Multiplica todo el espacio de estados existente por el rango de valores de temperatura, aumentando exponencialmente los requerimientos de memoria para la tabla de políticas.

#### 2. Supuesto de observabilidad completa (MDP vs POMDP)
*   **Razonabilidad:** Es poco razonable asumir observabilidad completa en el mundo exterior urbano. Variables reales como micro-corrientes de viento locales, estado mecánico interno microscópico de los rotores, o una rama de árbol a punto de caer, **no son directamente observables** por las cámaras o sensores del dron.
*   **Impacto de usar MDP vs POMDP:** Al usar un modelo MDP estándar, estas variables ocultas se absorben ingenuamente como simple "ruido aleatorio" en la función de transición estocástica. Esto hace que el agente no pueda razonar sobre la incertidumbre (ej. "no sé si hay viento fuerte, mejor vuelo bajo"). Un modelo **POMDP** (Partially Observable MDP) permitiría al drone mantener una *creencia* (belief state) sobre variables ocultas (inferiendo el viento basado en movimientos pasados), siendo mucho más preciso y robusto, pero astronómicamente más difícil de calcular.

#### 3. Tarea Episódica vs Continua
*   **Clasificación:** El problema debe modelarse como una **tarea episódica**.
*   **Argumento:** Existe un final natural de la misión bien definido (estado terminal): el drone completa la ruta y regresa a su base para apagarse, recargar y esperar el siguiente paquete.
*   **Efecto en el diseño:** Al ser episódico, la función de recompensa no necesita estar diseñada para obligar al agente a promediar ganancias infinitas. Podemos simplemente usar recompensas terminales acumulativas (+100 al final). Además, la decisión del valor de $\gamma$ cambia: mientras que en tareas continuas es *obligatorio* que $\gamma < 1$ para que las sumas matemáticas no diverjan al infinito, en una tarea episódica de longitud finita es matemáticamente seguro e incluso ventajoso usar **$\gamma = 1$**, dándole exactamente el mismo peso al éxito final sin importar si tomó 12 o 13 pasos.